<a href="https://colab.research.google.com/github/Niraj-Bhatta/RAG-using-langchain/blob/main/rag_yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Rag project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

from google.colab import userdata
api_key = userdata.get('OpenAIKey')


In [3]:
%pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [4]:
!pip install -q langchain-huggingface sentence-transformers langchain-community faiss-cpu

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

# Using HuggingFace exclusively instead of OpenAI
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_11859/2600985353.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


#step 1a -indexing(Document INgestion)

In [7]:
with open("/content/drive/MyDrive/RAG/Project-1/transcript.txt", "r", encoding="utf-8") as file:
    transcript = file.read()

print(transcript[:500])

Chapter 1: Introduction to advice
0:011 second[Music]
0:088 secondsall right last question three pieces of advice to entrepreneurs what are your three pieces I'm an entrepreneur so you can share them with me okay what would
0:1515 secondsyou what would you how would you advise me well um I first of all I really need to take some thought to like how can I provide
0:2424 secondsadvice that would be most helpful and I'm not sure I've given enough thought to to to that to give you the best possible 


In [8]:
print("Total characters:", len(transcript))

Total characters: 2075


In [9]:
from langchain_core.documents import Document

document = Document(
    page_content=transcript,
    metadata={
        "source": "YouTube",
        "video_id": "CbAQUqnrDcA",
        "title": "Advice to Entrepreneurs"
    }
)

print(document)

page_content='Chapter 1: Introduction to advice
0:011 second[Music]
0:088 secondsall right last question three pieces of advice to entrepreneurs what are your three pieces I'm an entrepreneur so you can share them with me okay what would
0:1515 secondsyou what would you how would you advise me well um I first of all I really need to take some thought to like how can I provide
0:2424 secondsadvice that would be most helpful and I'm not sure I've given enough thought to to to that to give you the best possible answer but I think
0:3232 secondsum I think certainly uh being focused on something that your confident will have high value to someone else um and just
Chapter 2: Value and realism
0:4141 secondsbeing really rigorous in making that assessment um because people are T tend to a natural human tendency is wishful
0:4949 secondsthinking um so a challenge for entrepreneurs is to say well what's the difference between really believing in your ideals and sticking sticking to
Chapter 3: Ri

#step 1b indexing(text splitting)

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents([document])

print("Total chunks:", len(chunks))

Total chunks: 6


In [11]:
len(chunks)

6

In [12]:
chunks[0]

Document(metadata={'source': 'YouTube', 'video_id': 'CbAQUqnrDcA', 'title': 'Advice to Entrepreneurs'}, page_content="Chapter 1: Introduction to advice\n0:011 second[Music]\n0:088 secondsall right last question three pieces of advice to entrepreneurs what are your three pieces I'm an entrepreneur so you can share them with me okay what would\n0:1515 secondsyou what would you how would you advise me well um I first of all I really need to take some thought to like how can I provide")

#step 1c and 1d

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
# Check the number of stored vectors
print("Stored vectors:", vector_store.index.ntotal)

# Check the embedding dimensions
print("Embedding dimensions:", vector_store.index.d)

# Check the number of document chunks
print("Document chunks:", len(vector_store.docstore._dict))

Stored vectors: 6
Embedding dimensions: 384
Document chunks: 6


##Step 2 Retrievel

In [15]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(
    "What advice does he give to entrepreneurs?"
)

for result in results:
    print(result.page_content)
    print(result.metadata)

Chapter 1: Introduction to advice
0:011 second[Music]
0:088 secondsall right last question three pieces of advice to entrepreneurs what are your three pieces I'm an entrepreneur so you can share them with me okay what would
0:1515 secondsyou what would you how would you advise me well um I first of all I really need to take some thought to like how can I provide
{'source': 'YouTube', 'video_id': 'CbAQUqnrDcA', 'title': 'Advice to Entrepreneurs'}
0:4949 secondsthinking um so a challenge for entrepreneurs is to say well what's the difference between really believing in your ideals and sticking sticking to
Chapter 3: Rigorous self-analysis
0:5858 secondsthem versus pursuing some unrealistic dream that doesn't actually have Merit and it's it's that is a that is a really
{'source': 'YouTube', 'video_id': 'CbAQUqnrDcA', 'title': 'Advice to Entrepreneurs'}
1:301 minute, 30 secondsgo and a lot of work that all those things improve the odds of success okay um I mean if if if other people are
1:

#3. Augmentation

In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Load local model and tokenizer using free local GPU resources
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    return_full_text=False
)

# Define the local HuggingFace model pipeline
llm = HuggingFacePipeline(pipeline=hf_pipe)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [17]:
prompt = PromptTemplate(
    template="""
    You are a helpful AI assistant. Use the following pieces of context to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum. Keep the answer as concise as possible.
    Always say "thanks for asking!" at the end of the answer.

    {context}

    Question : {question}
    """,
    input_variables=["context", "question"]
)



In [18]:
question = "is the topic of elon musk present here? if yes waht was it?"
retrieved_docs = retriever.invoke(question)

In [19]:
retrieved_docs

[Document(id='17e9c935-1368-4782-aa07-dcabb1656d07', metadata={'source': 'YouTube', 'video_id': 'CbAQUqnrDcA', 'title': 'Advice to Entrepreneurs'}, page_content="0:4949 secondsthinking um so a challenge for entrepreneurs is to say well what's the difference between really believing in your ideals and sticking sticking to\nChapter 3: Rigorous self-analysis\n0:5858 secondsthem versus pursuing some unrealistic dream that doesn't actually have Merit and it's it's that is a that is a really"),
 Document(id='db03da85-7b3d-4541-9292-2286fed26971', metadata={'source': 'YouTube', 'video_id': 'CbAQUqnrDcA', 'title': 'Advice to Entrepreneurs'}, page_content="0:2424 secondsadvice that would be most helpful and I'm not sure I've given enough thought to to to that to give you the best possible answer but I think\n0:3232 secondsum I think certainly uh being focused on something that your confident will have high value to someone else um and just\nChapter 2: Value and realism\n0:4141 secondsbeing real

In [20]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context" : context_text, "question" : question})

final_prompt

StringPromptValue(text='\n    You are a helpful AI assistant. Use the following pieces of context to answer the question.\n    If you don\'t know the answer, just say that you don\'t know.\n    Use three sentences maximum. Keep the answer as concise as possible.\n    Always say "thanks for asking!" at the end of the answer.\n\n    0:4949 secondsthinking um so a challenge for entrepreneurs is to say well what\'s the difference between really believing in your ideals and sticking sticking to\nChapter 3: Rigorous self-analysis\n0:5858 secondsthem versus pursuing some unrealistic dream that doesn\'t actually have Merit and it\'s it\'s that is a that is a really\n\n0:2424 secondsadvice that would be most helpful and I\'m not sure I\'ve given enough thought to to to that to give you the best possible answer but I think\n0:3232 secondsum I think certainly uh being focused on something that your confident will have high value to someone else um and just\nChapter 2: Value and realism\n0:4141 se

#step 4 - Generation

In [21]:
# Fallback configuration to make sure variables are defined in the session
if 'retriever' not in globals() and 'vector_store' in globals():
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

if 'question' not in globals():
    question = "is the topic of elon musk present here? if yes waht was it?"

if 'retrieved_docs' not in globals() and 'retriever' in globals():
    retrieved_docs = retriever.invoke(question)

if 'prompt' not in globals():
    from langchain_core.prompts import PromptTemplate
    prompt = PromptTemplate(
        template="""
        You are a helpful AI assistant. Use the following pieces of context to answer the question.
        If you don't know the answer, just say that you don't know.
        Use three sentences maximum. Keep the answer as concise as possible.
        Always say "thanks for asking!" at the end of the answer.

        {context}

        Question : {question}
        """,
        input_variables=["context", "question"]
    )

# Assemble context and prompt
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"context" : context_text, "question" : question})

# Generate final response
answer = llm.invoke(final_prompt)
print(answer)

 Elon Musk is mentioned in this conversation. The topic of Elon Musk is entrepreneurship. Specifically, he is discussed in relation to his challenges in focusing on realistic goals rather than unrealistic ones. He emphasizes the importance of being confident about providing high-value solutions to others while also practicing rigorous self-assessment. This aligns with Chapter 3 of the provided transcript which focuses on rigorous self-analysis and the distinction between idealism and practicality in entrepreneurial pursuits. Therefore, the answer is yes, Elon Musk is present in the discussion, specifically regarding his approach to entrepreneurship. Thanks for asking!


#These are the four steps explored here:

## 1.Indexing
## 2.Retrieval
## 3.Augmentation
##4.Generation